<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1404.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

In [ ]:
dataset = load_dataset('phatvo/hotpotqa-raft')

In [ ]:
# Функция-фильтр: возвращает True, если oracle_context есть в instruction
def has_oracle_in_instruction(example):
    return example['oracle_context'] in example['instruction']

# Применяем фильтрацию
dataset_with_oracle = dataset['train'].filter(has_oracle_in_instruction)
dataset_without_oracle = dataset['train'].filter(lambda ex: not has_oracle_in_instruction(ex))

print(f"Примеров с золотым документом: {len(dataset_with_oracle)}")
print(f"Примеров без золотого документа: {len(dataset_without_oracle)}")

In [ ]:
dataset_with_oracle = dataset_with_oracle.select(range(10))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

In [ ]:
def format_example(example):
    """
    Формирует текстовый промпт в формате чата Qwen2.5-Instruct.
    """
    # Создаём список сообщений, как того ожидает шаблон чата
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["cot_answer"]}
    ]
    # Применяем шаблон чата — он добавит все необходимые служебные токены
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  # Пока не токенизируем, оставляем текстом
        add_generation_prompt=False  # Не добавляем маркер начала ответа, т.к. ответ уже есть
    )
    return {"formatted_text": formatted}

In [ ]:
# Применяем функцию ко всему датасету
dataset_formatted = dataset_with_oracle.map(format_example)

In [ ]:
dataset_formatted

In [ ]:
def tokenize_function(examples):
    """
    Токенизирует столбец 'formatted_text' и создаёт labels.
    """
    # Токенизируем текст, обрезая до максимальной длины контекста модели (32768 для Qwen2.5-0.5B)
    # Мы используем 1024 для экономии памяти и ускорения обучения
    tokenized = tokenizer(
        examples["formatted_text"],
        truncation=True,
        padding=False,           # Padding будем делать динамически в DataCollator
        max_length=1024,
        # padding_side = 'right',
        truncation_side='left',  # Обрезаем токены слева
        return_tensors=None,     # Возвращаем обычные списки, не тензоры
    )
    return tokenized

In [ ]:
# Применяем токенизацию ко всему датасету
dataset_tokenized = dataset_formatted.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset_formatted.column_names,  # Удаляем старые текстовые столбцы для экономии памяти
)

print("Токенизация завершена. Пример размера:")
print(f"input_ids длина: {len(dataset_tokenized[0]['input_ids'])}")

In [ ]:
dataset_tokenized

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
# Настраиваем конфигурацию LoRA
lora_config = LoraConfig(
    r=8,                     # Ранг матриц адаптера (типичное значение — 8 или 16)
    lora_alpha=32,           # Коэффициент масштабирования (обычно вдвое больше r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Слои, к которым применяется LoRA для Qwen2.5
    lora_dropout=0.1,       # Dropout для регуляризации адаптеров
    bias="none",             # Не добавляем обучаемое смещение
    task_type=TaskType.CAUSAL_LM,  # Тип задачи — языковое моделирование
)

In [ ]:
!pip install -q triton

In [ ]:
# Применяем LoRA к модели
model = get_peft_model(model, lora_config)

In [ ]:
# Выведем информацию о количестве обучаемых параметров
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen-raft-lora",          # Папка для сохранения чекпоинтов
    num_train_epochs=3,                     # Количество проходов по датасету
    per_device_train_batch_size=2,          # Размер батча на одно устройство (GPU)
    gradient_accumulation_steps=2,          # Накапливаем градиенты 4 шага перед обновлением
    learning_rate=5e-5,                     # Скорость обучения (стандартное значение для LoRA)
    warmup_steps=5,                         # Шаги для постепенного увеличения LR в начале
    logging_steps=10,                       # Выводить лог каждые 10 шагов
    save_steps=100,                         # Сохранять чекпоинт каждые 100 шагов
    save_total_limit=2,                     # Хранить только последние 2 чекпоинта
    fp16=True,                              # Использовать смешанную точность (быстрее и меньше памяти)
    report_to="none",                       # Отключаем внешние трекеры (wandb, tensorboard)
    remove_unused_columns=False,            # Не удалять лишние столбцы датасета
)

In [ ]:
from transformers import Trainer, DataCollatorForLanguageModeling

In [ ]:
# Создаём сборщик батчей для задачи языкового моделирования
# Он автоматически дополнит последовательности токенов до одинаковой длины внутри батча
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Мы НЕ используем Masked Language Modeling, у нас Causal LM
)

In [ ]:
# Создаём тренер
trainer = Trainer(
    model=model,                     # Наша модель с LoRA
    args=training_args,              # Параметры обучения
    train_dataset=dataset_tokenized, # Токенизированный датасет
    data_collator=data_collator,     # Сборщик батчей
)

In [ ]:
# Запускаем обучение
trainer.train()